In [2]:
from pathlib import Path

# Always anchor paths to project root
PROJECT_ROOT = Path.cwd()

# If running inside notebooks/02_preprocess, go up two levels
if PROJECT_ROOT.name == "02_preprocess":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: c:\Users\User\Desktop\Vehicle_Damage_Detection


In [1]:
from pathlib import Path
from PIL import Image

# ===================== SETTINGS =====================
TARGET_SIZE = (224, 224)

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    while cur != cur.parent:
        if (cur / ".dvc").exists() or (cur / ".git").exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError("Could not find project root (no .dvc or .git found).")

PROJECT_ROOT = find_project_root(Path.cwd())

RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "CarDD_release" / "CarDD_release" / "CarDD_SOD"
OUT_ROOT = PROJECT_ROOT / "data" / "processed" / "cardd_sod"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT:", RAW_ROOT, "| exists:", RAW_ROOT.exists())
print("OUT_ROOT:", OUT_ROOT)

# ===================== SPLIT MAPPING (YOUR CASE) =====================
SPLIT_MAP = {
    "train": RAW_ROOT / "CarDD-TR",
    "val":   RAW_ROOT / "CarDD-VAL",
    "test":  RAW_ROOT / "CarDD-TE",
}

for k, v in SPLIT_MAP.items():
    print(f"{k.upper()} split path:", v, "| exists:", v.exists())

if not all(p.exists() for p in SPLIT_MAP.values()):
    raise FileNotFoundError("One of CarDD-TR / CarDD-VAL / CarDD-TE is missing. Check folder names again.")

# ===================== HELPERS =====================
def find_images_and_masks(split_dir: Path):
    """
    Inside CarDD-TR / CarDD-VAL / CarDD-TE
    find a folder that looks like images and one that looks like masks.
    """
    img_exts = {".jpg", ".jpeg", ".png"}
    mask_exts = {".png"}

    subdirs = [p for p in split_dir.iterdir() if p.is_dir()]
    if not subdirs:
        raise FileNotFoundError(f"No subfolders in {split_dir}")

    def count_ext(d, exts):
        return sum(1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in exts)

    # image folder = most jpg/jpeg/png
    img_dir = max(subdirs, key=lambda d: count_ext(d, img_exts))

    # mask folder = most png (try avoid same as img_dir)
    ranked = sorted(subdirs, key=lambda d: count_ext(d, mask_exts), reverse=True)
    mask_dir = ranked[0]
    if mask_dir == img_dir and len(ranked) > 1:
        mask_dir = ranked[1]

    return img_dir, mask_dir

def resize_image(src: Path, dst: Path):
    with Image.open(src) as im:
        im = im.convert("RGB")
        im = im.resize(TARGET_SIZE, resample=Image.BILINEAR)
        dst.parent.mkdir(parents=True, exist_ok=True)
        im.save(dst, format="JPEG", quality=95)

def resize_mask(src: Path, dst: Path):
    with Image.open(src) as m:
        m = m.resize(TARGET_SIZE, resample=Image.NEAREST)
        dst.parent.mkdir(parents=True, exist_ok=True)
        m.save(dst, format="PNG")

# ===================== RUN PREPROCESS =====================
OUT_ROOT.mkdir(parents=True, exist_ok=True)

total_img = 0
total_mask = 0

for split in ["train", "val", "test"]:
    split_dir = SPLIT_MAP[split]

    img_dir, mask_dir = find_images_and_masks(split_dir)

    print(f"\n--- {split.upper()} ---")
    print("Split dir :", split_dir)
    print("Images dir:", img_dir)
    print("Masks dir :", mask_dir)

    out_img_dir = OUT_ROOT / split / "images"
    out_mask_dir = OUT_ROOT / split / "masks"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_mask_dir.mkdir(parents=True, exist_ok=True)

    img_files = sorted([f for f in img_dir.iterdir()
                        if f.is_file() and f.suffix.lower() in [".jpg", ".jpeg", ".png"]])

    processed_img = 0
    processed_mask = 0
    missing_mask = 0

    for img_path in img_files:
        stem = img_path.stem

        # find corresponding mask by same stem
        mask_path = None
        for ext in [".png", ".jpg", ".jpeg"]:
            cand = mask_dir / f"{stem}{ext}"
            if cand.exists():
                mask_path = cand
                break

        out_img_path = out_img_dir / f"{stem}.jpg"
        out_mask_path = out_mask_dir / f"{stem}.png"

        resize_image(img_path, out_img_path)
        processed_img += 1

        if mask_path is None:
            missing_mask += 1
        else:
            resize_mask(mask_path, out_mask_path)
            processed_mask += 1

    print(" Images processed:", processed_img)
    print(" Masks processed :", processed_mask)
    print(" Missing masks   :", missing_mask)

    total_img += processed_img
    total_mask += processed_mask

print("\n.. PREPROCESS DONE")
print("Saved to:", OUT_ROOT)
print("Total images:", total_img)
print("Total masks :", total_mask)


PROJECT_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection
RAW_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_SOD | exists: True
OUT_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod
TRAIN split path: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_SOD\CarDD-TR | exists: True
VAL split path: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_SOD\CarDD-VAL | exists: True
TEST split path: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_SOD\CarDD-TE | exists: True

--- TRAIN ---
Split dir : C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_SOD\CarDD-TR
Images dir: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_SOD\CarDD-TR\CarDD-TR-Edge
Masks dir : C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD

In [2]:
from pathlib import Path

p = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod")

def count_files(folder, exts):
    return sum(1 for f in folder.rglob("*") if f.is_file() and f.suffix.lower() in exts)

for split in ["train", "val", "test"]:
    img_dir = p / split / "images"
    mask_dir = p / split / "masks"
    print(f"\n{split.upper()}")
    print(" images:", count_files(img_dir, {".jpg", ".jpeg", ".png"}))
    print(" masks :", count_files(mask_dir, {".png"}))



TRAIN
 images: 2816
 masks : 2816

VAL
 images: 810
 masks : 810

TEST
 images: 374
 masks : 374
